# RQ2-v3 four-fold held-out variance gate

CPU-only analysis of 16 saved per-batch interior-gradient Gram matrices. Folds follow `batch_id % 4`; each Gradient-Oracle-Marginal policy is fit on 12 batches and evaluated on four unseen batches. Geometry and Resource policies stay frozen. Variance is evaluated exactly over all 91 pairs—no Monte Carlo pair sampling, accuracy, test data, or network training.

In [ ]:
import os, subprocess, sys, json, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
GIT_COMMIT = subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('Commit:', GIT_COMMIT)
print('CPU-only notebook; accelerator should be None')

## Locate an interaction probe containing 16 per-batch Gram matrices

In [ ]:
import importlib, numpy as np
import rq2_gradient_variance_v3, rq2_cross_batch_variance_4fold
rq2_gradient_variance_v3 = importlib.reload(rq2_gradient_variance_v3)
rq2_cross_batch_variance_4fold = importlib.reload(rq2_cross_batch_variance_4fold)
INTERACTION_ROOT = rq2_gradient_variance_v3.find_interaction_probe_root(
    Path('/kaggle/input'), '/kaggle/working/materialized-rq2-cross-batch-4fold'
)
GRAM_PATH = INTERACTION_ROOT/'gram_matrices.npy'
assert GRAM_PATH.is_file(), (
    'This older interaction output lacks per-batch Grams. Rerun the updated '
    'kaggle_rq2_cross_subnet_interaction.ipynb once; no 100-epoch training is needed.'
)
grams = np.load(GRAM_PATH, mmap_mode='r')
assert grams.shape == (16,14,14), grams.shape
print('Interaction root:', INTERACTION_ROOT)
print('Gram tensor:', GRAM_PATH, grams.shape, grams.dtype)

## Run deterministic 4-fold cross-batch validation

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/v3_cross_batch_variance')
result = rq2_cross_batch_variance_4fold.run_four_fold_cross_batch_validation(
    INTERACTION_ROOT, OUTPUT_DIR, bootstrap_draws=10_000
)
result['git_commit'] = GIT_COMMIT
(OUTPUT_DIR/'metadata.json').write_text(json.dumps(result, indent=2)+'\n')
print(json.dumps(result, indent=2))

## Inspect primary gate, uncertainty, and oracle stability

In [ ]:
import pandas as pd
from IPython.display import display, Image
display(pd.read_csv(OUTPUT_DIR/'fold_variance.csv'))
display(pd.read_csv(OUTPUT_DIR/'summary.csv'))
display(pd.read_csv(OUTPUT_DIR/'geo_resource_paired_uncertainty.csv'))
display(pd.read_csv(OUTPUT_DIR/'oracle_marginal_stability.csv'))
display(pd.read_csv(OUTPUT_DIR/'heldout_geometry_moment_bridge.csv'))
display(Image(filename=str(OUTPUT_DIR/'variance_by_fold.png')))
display(Image(filename=str(OUTPUT_DIR/'geo_vs_resource_batch_delta.png')))
display(Image(filename=str(OUTPUT_DIR/'oracle_marginal_stability.png')))

## Validate and export

In [ ]:
required = [
    'metadata.json','gram_matrices.npy','fold_assignment.csv',
    'policy_uniform.csv','policy_resource.csv','policy_geometry.csv',
    'oracle_policy_fold0.csv','oracle_policy_fold1.csv','oracle_policy_fold2.csv','oracle_policy_fold3.csv',
    'batch_variance.csv','fold_variance.csv','summary.csv',
    'bootstrap_geo_minus_resource.npy','geo_resource_paired_uncertainty.csv',
    'oracle_marginal_stability.csv','heldout_geometry_moment_bridge.csv',
    'variance_by_fold.png','geo_vs_resource_batch_delta.png','oracle_marginal_stability.png',
]
missing = [name for name in required if not (OUTPUT_DIR/name).is_file() or (OUTPUT_DIR/name).stat().st_size == 0]
assert not missing, f'Missing four-fold artifacts: {missing}'
saved = json.loads((OUTPUT_DIR/'metadata.json').read_text())
assert saved['num_folds'] == 4 and saved['oracle_fit_only_on_training_folds'] is True
assert saved['network_training'] is False and saved['test_used'] is False
bundle_path = Path('/kaggle/working/v3_cross_batch_variance.zip')
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in OUTPUT_DIR.rglob('*'):
        if path.is_file(): bundle.write(path, path.relative_to(OUTPUT_DIR))
print('Download/persist:', bundle_path, f'{bundle_path.stat().st_size/2**20:.1f} MiB')
bundle_path